In [12]:
import os
from dotenv import load_dotenv
import openai
import anthropic
from google import genai  # Google'ın resmi kütüphanesi
from google.genai import types

# .env dosyasını yükle
load_dotenv()

# --- KRİTİK ÇATIŞMA ÇÖZÜMÜ ---
# Eğer sistemde eski bir GOOGLE_API_KEY varsa, onu Python hafızasından tamamen siliyoruz
if "GOOGLE_API_KEY" in os.environ:
    os.environ.pop("GOOGLE_API_KEY")

# Şimdi .env dosendeki güncel GEMINI_API_KEY'i alıyoruz
gemini_api_key = os.getenv("GEMINI_API_KEY")

# İstemcilerin (Client) tanımlanması
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
# Yeni anahtarı buraya açıkça veriyoruz
gemini_client = genai.Client(api_key=gemini_api_key) 

test_text = "Apple Inc. reported record quarterly earnings today."
prompt = f"Classify this news into one of these categories: World, Sports, Business, Technology. Text: {test_text}. Reply with only the category name."

# OpenAI test
oai_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)
print("OpenAI:", oai_response.choices[0].message.content.strip())

# Anthropic test
ant_response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=10,
    messages=[{"role": "user", "content": prompt}]
)
print("Anthropic:", ant_response.content[0].text.strip())


# Gemini test
# Gemini test
try:
    gem_response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=20,  # Token sınırını 20'ye çıkardık (Güvenlik payı)
            temperature=0.0,
            thinking_config=types.ThinkingConfig(thinking_budget=0) # Düşünmeyi tamamen kapatıyoruz!
        )
    )
    print("Gemini:", gem_response.text.strip() if gem_response.text else "Unknown")
except Exception as e:
    print("Gemini API Hatası:", e)

OpenAI: Business
Anthropic: Business
Gemini: Business
